# VadCLIP Subclass từ video description
Dùng bank 78 subclass abnormal có số lượng biến thiên theo class và giữ `Normal` dưới dạng một prompt. Subclass được gộp tại từng snippet trước, sau đó Temporal Top-K mới chạy trên 14 class như VadCLIP gốc. Chạy bốn cấu hình scratch/checkpoint kết hợp max/soft; không dùng test set để chọn epoch.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import shutil
import subprocess
import sys
import torch

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
SUBCLASS_JSON = PROJECT_ROOT / 'code' / 'ucf_subclass_bank_description_other.json'
SELF_TRAINED_BASELINE = SRC_DIR / 'model' / 'model_ucf_paper_scratch.pth'
TRAIN_LIST = LIST_DIR / 'ucf_CLIP_rgb_relative.csv'
TEST_LIST = LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'
EVALUATE_ROOT = PROJECT_ROOT / 'code' / 'results' / 'evaluate' / 'subclass'
BASELINE_CACHE = EVALUATE_ROOT / 'baseline_self_trained_metrics.json'
RUN_VARIANTS = ['scratch_max', 'scratch_soft', 'checkpoint_max', 'checkpoint_soft']
RUN_TRAIN = True
RUN_EVALUATION = True

print('PyTorch:', torch.__version__, '| CUDA build:', torch.version.cuda, '| GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime > Change runtime type > GPU, rồi kiểm tra PyTorch CUDA trước khi chạy.')
print('GPU:', torch.cuda.get_device_name(0))


## Chuẩn bị thư viện, feature và kiểm tra đầu vào
Chỉ nạp checkpoint ở hai cấu hình `checkpoint_*`; không dùng `model_ucf.pth` của tác giả.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'tqdm', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    archives = [PROJECT_ROOT / name for name in ('UCFClipFeatures.zip', 'UCFClipFeatures.tar', 'UCFClipFeatures.tar.gz')]
    archive = next((path for path in archives if path.is_file()), None)
    if archive is None:
        raise FileNotFoundError('Không tìm thấy /content/UCFClipFeatures hoặc archive trên Drive.')
    local_archive = Path('/content') / archive.name
    if not local_archive.is_file() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)
if not FEATURE_ROOT.is_dir():
    raise FileNotFoundError('Archive phải chứa thư mục gốc UCFClipFeatures/.')
required = [SUBCLASS_JSON, SELF_TRAINED_BASELINE, TRAIN_LIST, TEST_LIST,
            LIST_DIR / 'gt_ucf.npy', LIST_DIR / 'gt_segment_ucf.npy', LIST_DIR / 'gt_label_ucf.npy',
            SRC_DIR / 'model.py', SRC_DIR / 'ucf_evaluate.py', SRC_DIR / 'model_subclass.py',
            SRC_DIR / 'ucf_train_subclass.py', SRC_DIR / 'ucf_evaluate_subclass.py',
            SRC_DIR / 'utils' / 'tools.py', SRC_DIR / 'utils' / 'ucf_detectionMAP.py']
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Thiếu file trên Drive:\n' + '\n'.join(missing))
with TRAIN_LIST.open(encoding='utf-8', newline='') as f:
    train_count = sum(1 for _ in csv.DictReader(f))
with TEST_LIST.open(encoding='utf-8', newline='') as f:
    test_count = sum(1 for _ in csv.DictReader(f))
if (train_count, test_count) != (16100, 290):
    raise ValueError(f'List không đầy đủ: train={train_count}, test={test_count}')
sys.path.insert(0, str(SRC_DIR))
from model_subclass import load_subclass_bank, subclass_group_mask
groups = load_subclass_bank(SUBCLASS_JSON)
group_counts = [len(group) for group in groups]
group_mask = subclass_group_mask(groups)
BANK_TAG = SUBCLASS_JSON.stem.removeprefix('ucf_subclass_bank_')
AGGREGATION_ORDER_TAG = 'aggregate_first'
print('Train chunks:', train_count, '| Test videos:', test_count, '| Class groups:', len(groups))
print('Subclass counts:', dict(zip(('Normal', 'Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism'), group_counts)))
print('Total abnormal subclasses:', sum(group_counts[1:]), '| K_max:', group_mask.shape[1])
print('Feature root:', FEATURE_ROOT)
print('Self-trained baseline:', SELF_TRAINED_BASELINE)


## Smoke test một batch
Chạy forward/backward thực với CLIP pretrained trước khi bắt đầu bốn lượt train; không lưu checkpoint hoặc dùng nhãn test.

In [ ]:
from ucf_train_subclass import RelativeTrainDataset, make_model, classifier_mil_loss, class_mil_loss
from model_subclass import subclass_group_mask, text_regularization
normal_sample = RelativeTrainDataset(TRAIN_LIST, FEATURE_ROOT, True, 256)[0]
abnormal_sample = RelativeTrainDataset(TRAIN_LIST, FEATURE_ROOT, False, 256)[0]
visual = torch.stack([normal_sample[0], abnormal_sample[0]]).cuda()
labels = torch.tensor([normal_sample[1], abnormal_sample[1]], device='cuda')
lengths = torch.tensor([normal_sample[2], abnormal_sample[2]], device='cuda')
smoke_model = make_model('cuda').cuda()
class_text, logits1, subclass_logits = smoke_model(visual, None, groups, lengths)
loss_cls = classifier_mil_loss(logits1, labels, lengths)
smoke_mask = subclass_group_mask(groups, 'cuda')
loss_mil, scores = class_mil_loss(subclass_logits, labels, lengths, 'soft', 1.0, smoke_mask)
loss_text = text_regularization(class_text)
total = loss_cls + loss_mil + loss_text
if not torch.isfinite(total):
    raise FloatingPointError('Smoke loss không hữu hạn')
total.backward()
max_subclasses = max(len(group) for group in groups)
assert subclass_logits.shape == (2, 256, 14, max_subclasses)
assert scores.shape == (2, 14, max_subclasses)
assert torch.isneginf(subclass_logits[:, :, 0, 1:]).all(), 'Padding của Normal chưa được mask'
assert smoke_model.classifier.weight.grad is not None
assert smoke_model.text_prompt_embeddings.weight.grad is not None
assert all(parameter.grad is None for parameter in smoke_model.clipmodel.parameters())
print('Smoke test OK:', 'subclass_logits', tuple(subclass_logits.shape), 'loss', float(total))
del smoke_model, visual, logits1, subclass_logits, class_text, scores, total
torch.cuda.empty_cache()


## Chạy bốn cấu hình
Scratch chạy 10 epoch; checkpoint chạy 3 epoch. Cả hai dùng LR khởi đầu `2e-5`. Mỗi epoch đều lưu `.pth`; chỉ epoch cuối được báo làm kết quả chính. Checkpoint với LR mới được lưu riêng, không ghi đè kết quả cũ.

In [ ]:
EVALUATE_ROOT.mkdir(parents=True, exist_ok=True)
valid_variants = {'scratch_max', 'scratch_soft', 'checkpoint_max', 'checkpoint_soft'}
if not RUN_VARIANTS or any(name not in valid_variants for name in RUN_VARIANTS):
    raise ValueError(f'RUN_VARIANTS chỉ được lấy từ {sorted(valid_variants)}')
for name in RUN_VARIANTS:
    initialization, aggregation = name.split('_')
    epochs = 10 if initialization == 'scratch' else 3
    result_name = f'{name}_{BANK_TAG}_{AGGREGATION_ORDER_TAG}'
    output_dir = SRC_DIR / 'model' / f'epoch_checkpoints_subclass_{result_name}'
    final_model = output_dir / f'epoch_{epochs:03d}.pth'
    eval_dir = EVALUATE_ROOT / result_name
    print('\n' + '=' * 70 + '\nVariant:', name)
    if RUN_TRAIN:
        if final_model.is_file():
            config = json.loads((output_dir / 'config.json').read_text(encoding='utf-8'))
            current_bank_hash = hashlib.sha256(SUBCLASS_JSON.read_bytes()).hexdigest()
            expected = (initialization, aggregation, current_bank_hash, 2e-5, 'subclass_then_temporal_topk')
            actual = (config['initialization'], config['aggregation'], config['subclass_sha256'], config['resolved_lr'], config.get('aggregation_order'))
            if actual != expected:
                raise ValueError(f'Checkpoint đã có nhưng không khớp cấu hình hoặc subclass bank: {output_dir}')
            print('Đã có final checkpoint đúng cấu hình, bỏ qua train:', final_model)
        else:
            train_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_train_subclass.py'),
                         '--subclass-json', str(SUBCLASS_JSON), '--feature-root', str(FEATURE_ROOT),
                         '--train-list', str(TRAIN_LIST), '--initialization', initialization,
                         '--aggregation', aggregation, '--temperature', '1.0',
                         '--output-dir', str(output_dir), '--seed', '234', '--lr', '2e-5',
                         '--batch-size', '64', '--num-workers', '2']
            if initialization == 'checkpoint':
                train_cmd += ['--pretrained-model-path', str(SELF_TRAINED_BASELINE)]
            subprocess.run(train_cmd, cwd=SRC_DIR, check=True)
    if RUN_EVALUATION:
        if not final_model.is_file():
            raise FileNotFoundError(f'Chưa có checkpoint epoch cố định: {final_model}')
        eval_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_subclass.py'),
                    '--subclass-json', str(SUBCLASS_JSON), '--feature-root', str(FEATURE_ROOT),
                    '--test-list', str(TEST_LIST), '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
                    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
                    '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
                    '--baseline-model-path', str(SELF_TRAINED_BASELINE),
                    '--baseline-metrics-json', str(BASELINE_CACHE),
                    '--model-path', str(final_model), '--aggregation', aggregation,
                    '--temperature', '1.0', '--output-dir', str(eval_dir)]
        eval_result = subprocess.run(
            eval_cmd, cwd=SRC_DIR, text=True, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, bufsize=1
        )
        print(eval_result.stdout, end='')
        if eval_result.returncode != 0:
            raise RuntimeError(
                f'Evaluation thất bại với exit code {eval_result.returncode}. '
                'Traceback gốc đã được in ngay phía trên.'
            )
        print('Evaluation saved:', eval_dir)


## Bảng tổng hợp
Chỉ đọc kết quả đã lưu; không chạy train hoặc đánh giá lại.

In [ ]:
columns = ['classifier_auc', 'classifier_ap', 'alignment_auc', 'alignment_ap', 'ano_auc', 'mAP@0.1', 'mAP@0.2', 'mAP@0.3', 'mAP@0.4', 'mAP@0.5', 'avg_mAP']
rows = []
for name in sorted(valid_variants):
    result_name = f'{name}_{BANK_TAG}_{AGGREGATION_ORDER_TAG}'
    metrics_path = EVALUATE_ROOT / result_name / 'metrics.json'
    if metrics_path.is_file():
        data = json.loads(metrics_path.read_text(encoding='utf-8'))
        if not rows:
            rows.append({'variant': 'baseline_self_trained', **{key: data['baseline'][key] for key in columns}})
        rows.append({'variant': result_name, **{key: data['subclass'][key] for key in columns}})
if not rows:
    print('Chưa có evaluation nào.')
else:
    print('variant,' + ','.join(columns))
    for row in rows:
        print(row['variant'] + ',' + ','.join(f'{row[key]:.2f}' for key in columns))
    summary_path = EVALUATE_ROOT / f'comparison_{BANK_TAG}_{AGGREGATION_ORDER_TAG}.csv'
    with summary_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['variant', *columns])
        writer.writeheader()
        writer.writerows(rows)
    print('Saved:', summary_path)
